[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/colab_pipeline_demo.ipynb)

# Pipeline demo — `analyze_video` on the bake-off clip (Phase 3)

Runs the full soccer-vision pipeline end-to-end:
detection + tracking + per-frame pitch homography → enriched per-detection
trajectories in canonical pitch coords + per-frame possession/phase labels.

Writes four parquet files under `/content/out/`:
- `trajectories_px.parquet`, `keypoints.parquet` — Stage-1 checkpoint (raw tracker output)
- `trajectories.parquet`, `phases.parquet` — the deliverables

**Prereqs**
- GPU runtime (Runtime → Change runtime type → GPU).
- `bakeoff_clip.mp4` at `MyDrive/soccer-vision/`.
- The fine-tuned ball model auto-downloads from the `ball-v1` GitHub release;
  the player + pitch models download from Drive on first run.

This notebook doubles as the **§4.1 calibration check**: confirm projected pitch
coords land in `[0,1]²` and the two teams separate across the goal-to-goal (y) axis.
If they don't, the length/width axis convention in `pitch/landmarks.py` or the
`{0:own, 1:opp}` cluster mapping in `tracking/roboflow.py` needs a flip.

In [ ]:
# Install the package + roboflow extra (ultralytics, supervision, sports, gdown).
# Re-runnable clone; non-editable install (avoids PEP 660 editable-metadata issues on Colab).
!rm -rf /content/soccer-vision
!git clone -q https://github.com/PatrickJReed/soccer-vision.git /content/soccer-vision
!pip install -q "/content/soccer-vision/packages/soccer-vision[roboflow]"

from google.colab import drive
drive.mount('/content/drive')

# If a later cell raises ImportError after this install, do Runtime -> Restart session
# once and re-run from this cell (ultralytics can pull a numpy/torch that needs a restart).

In [ ]:
from pathlib import Path
from soccer_vision import analyze_video

CLIP = Path("/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4")
OUT = Path("/content/out")
assert CLIP.exists(), f"Place bakeoff_clip.mp4 at {CLIP}"

# backend=None -> RoboflowBackend(detect_pitch=True); weights auto-download on first run.
result = analyze_video(CLIP, OUT)

print(f"homography_coverage: {result.homography_coverage:.1%}  (frames the pitch model fit an H)")
print(f"ball_coverage:       {result.ball_coverage:.1%}  (frames with a ball pitch coord)")

In [ ]:
import pandas as pd

print("Possession states (per frame):")
print(result.phases["possession_state"].value_counts(), "\n")
print("Phases (per frame):")
print(result.phases["phase"].value_counts(), "\n")
print("Output files:")
for p in sorted(OUT.glob("*.parquet")):
    print(f"  {p.name:24s} {len(pd.read_parquet(p)):>7d} rows")

## §4.1 calibration check

If the homography + axis convention are right:
- the vast majority of mapped player detections fall inside `[0,1]²`, and
- the two teams separate along **y** (one team's mean y below the other's),
  i.e. each team mostly defends its own half.

The scatter shows the best-covered frame on the canonical pitch (y is goal-to-goal;
dotted lines mark the defensive/attacking thirds at 0.333 / 0.667).

In [ ]:
import matplotlib.pyplot as plt

traj = result.trajectories
players = traj[traj["class"].isin(["player", "goalkeeper"])].dropna(subset=["x_pitch", "y_pitch"])

inside = players[players["x_pitch"].between(0, 1) & players["y_pitch"].between(0, 1)]
print(f"mapped player detections: {len(players)}")
print(f"  inside [0,1]^2: {len(inside) / max(1, len(players)):.1%}")
print("mean y_pitch by team (should separate):")
print(players.groupby("team")["y_pitch"].mean())

best_frame = int(players["frame"].value_counts().idxmax())
f = traj[traj["frame"] == best_frame]
plt.figure(figsize=(6, 4))
for team, color in [("own", "tab:blue"), ("opp", "tab:red")]:
    s = f[f["class"].isin(["player", "goalkeeper"]) & (f["team"] == team)]
    plt.scatter(s["x_pitch"], s["y_pitch"], c=color, label=team, s=60)
ball = f[f["class"] == "ball"]
plt.scatter(ball["x_pitch"], ball["y_pitch"], c="black", marker="*", s=250, label="ball")
plt.axhline(0.333, ls=":", c="gray")
plt.axhline(0.667, ls=":", c="gray")
plt.xlim(-0.1, 1.1)
plt.ylim(-0.1, 1.1)
plt.gca().invert_yaxis()
plt.xlabel("x_pitch")
plt.ylabel("y_pitch (goal-to-goal)")
plt.title(f"Frame {best_frame} — canonical pitch")
plt.legend()
plt.tight_layout()
plt.show()